In [1]:
# =========================================================
# AUDIO → TEXT → TRANSLATION (LangGraph + Hugging Face) → VOICE OUTPUT
# =========================================================

In [24]:
# 1️⃣ Imports
import os, json, re
from pathlib import Path
from dotenv import load_dotenv
from gtts import gTTS
from transformers import pipeline
import torch
import tiktoken
from openai import OpenAI
from IPython.display import Audio, display
from nbformat import read
from nbconvert import PythonExporter
import time

In [3]:
#!pip install gTTS
#!pip install nbformat
#!pip install nbconvert

In [4]:
# 2️⃣ Load environment safely
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

In [6]:
# 3️⃣ Helper: Import functions and variables from your text translation notebook
def import_notebook_functions(nb_path: str):
    """Import all functions and variables from another Jupyter notebook."""
    
    with open(nb_path, "r", encoding="utf-8") as f:
        nb = read(f, as_version=4)
    
    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    
    # Disable interactive input() lines safely
    safe_source = []
    for line in source.splitlines():
        if "input(" in line:  # Skip user input lines
            safe_source.append("# " + line)
        else:
            safe_source.append(line)
    safe_source = "\n".join(safe_source)
    
    exec(source, globals())

# Import from your existing LangGraph notebook
import_notebook_functions("real_time_text_translation.ipynb")


💬 Enter your text to translate:  aaa
🌍 Enter target language code (like 'ta' for Tamil, 'fr' for French):  ta


Cleaned Text: aaa
Detected Language: en
Tone: Neutral tone
Moderation: Safe
Translation: It seems like "aaa" is not a word or phrase that can be translated. If you have a specific text or phrase you'd like to translate into Tamil, please provide that, and I'll be happy to help!


In [7]:
# 4️⃣ Function: Transcribe audio → text using Whisper (OpenAI)
def transcribe_audio_openai(file_path: str) -> str:
    """Convert audio to text using Whisper API."""
    with open(file_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="gpt-4o-mini-transcribe",
            file=audio_file
        )
    return transcript.text.strip()

In [8]:
# 5️⃣ Function: Count tokens and estimate cost
def count_tokens(text, model="gpt-4o-mini"):
    """Count tokens and calculate estimated cost."""
    enc = tiktoken.encoding_for_model(model)
    num_tokens = len(enc.encode(text))
    price_per_1k = 0.00015  # USD per 1K tokens (input or output)
    cost = (num_tokens / 1000) * price_per_1k
    return num_tokens, cost

In [35]:
# 6️⃣ Main pipeline: Audio → Text → LangGraph Translation → Voice Output
import os, json
from IPython.display import Audio

def audio_to_translated(audio_file_path: str, target_language: str = "ta"):
    """
    Complete workflow:
    1. Transcribe audio to text
    2. Run LangGraph translation
    3. Generate translated audio with Amazon Polly
    4. Return translated text, cost summary, and audio
    """

    # Step 1: Verify file path
    if not os.path.exists(audio_file_path):
        print(f"File not found: {audio_file_path}")
        return None, None, None

    print("Processing audio file:", audio_file_path)

    # Step 2: Restrict file size (max 25 MB)
    max_size_mb = 16
    file_size_mb = os.path.getsize(audio_file_path) / (1024 * 1024)
    if file_size_mb > max_size_mb:
        print(f"File too large ({file_size_mb:.2f} MB). Please use a file under {max_size_mb} MB.")
        return None, None, None

    print(f"\nProcessing audio file: {audio_file_path} ({file_size_mb:.2f} MB)")

    # Step 3: Transcription
    try:
        text = transcribe_audio_openai(audio_file_path).strip()
        print("Transcribed Text:", text)
    except Exception as e:
        print(f"Transcription failed: {e}")
        return None, None, None

    #Language Detection
    try:
        detected_lang = detect(text)
        print("Detected Language:", detected_lang)
    except:
        detected_lang = "unknown"
        print("Language detection failed")
        
    # Step 4: Translation using LangGraph
    try:
        initial_state = {"text": text, "target_language": target_language}
        final_state = app.invoke(initial_state)
        translated_text = final_state.get("translated_text", "").strip()
        print("Translated Text:", translated_text)
    except Exception as e:
        print(f"Translation failed: {e}")
        return None, None, None

    # Step 5: Post-processing (cleanup)
    translated_text = translated_text.replace("\n", " ").strip()

    '''
     #AWS Poly Cloud
    import boto3
    def text_to_speech_polly(text, output_path="translated_audio.mp3", language_code="ta-IN", voice_id="Aditi"):
        """Convert translated text to audio using AWS Polly."""
        try:
            # Initialize Polly client
            polly_client = boto3.client(
                "polly",
                region_name="us-east-1",
                aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
                aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY")
            )
    
            # Request speech synthesis
            response = polly_client.synthesize_speech(
                Text=text,
                OutputFormat="mp3",
                VoiceId=voice_id,
                LanguageCode=language_code
            )
    
            # Save audio stream
            with open(output_path, "wb") as f:
                f.write(response["AudioStream"].read())
    
            print(f"Audio saved at: {output_path}")
            return output_path
    
        except Exception as e:
            print(f"TTS failed: {e}")
            return None 
    '''
    #Step 6 : Convert text to speech using gTTS (Google Text-to-Speech)
    def text_to_speech_gtts(text, output_path="translated_audio.mp3", lang_code="ta"):
        """Convert translated text to audio using gTTS."""
        try:
            tts = gTTS(text=text, lang=lang_code)
            tts.save(output_path)
            print(f"Audio saved at: {output_path}")
            return output_path
        except Exception as e:
            print(f"TTS failed: {e}")
            return None
    
    
    '''# Step 7: Convert text to audio (Amazon Polly or gTTS)
    try:
        output_dir = "audio_output"
        os.makedirs(output_dir, exist_ok=True)  # create folder if not exists
        output_path = os.path.join(output_dir, "translated_audio.mp3")
    
        audio_out_path = text_to_speech_polly(translated_text, output_path=output_path)
    except Exception as e:
        print(f"Speech synthesis failed: {e}")
        audio_out_path = None'''
    
    # Step 7: Convert to speech
    try:
        os.makedirs("audio_output", exist_ok=True)
        output_path = os.path.join("audio_output", Path(audio_file_path).stem + "_translated.mp3")
        audio_out_path = text_to_speech_gtts(translated_text, output_path, lang_code=target_language)
    except Exception as e:
        print(f"Speech synthesis failed: {e}")
        audio_out_path = None

    # Step 8: Token count and pricing
    input_tokens = len(text.split())
    output_tokens = len(translated_text.split())
    total_tokens = input_tokens + output_tokens

    # Pricing: OpenAI gpt-4o-mini + AWS Polly
    openai_cost = (total_tokens / 1000) * 0.00015
    aws_cost = len(translated_text) * 0.000004

    summary = {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "openai_cost_usd": round(openai_cost, 6),
        "aws_cost_usd": round(aws_cost, 6),
        "translated_text": translated_text,
        "audio_out_path": audio_out_path,
    }

    #print("\nSUMMARY:", json.dumps(summary, indent=2, ensure_ascii=False))

    if audio_out_path and os.path.exists(audio_out_path):
        return translated_text, summary, Audio(audio_out_path)
    else:
        return translated_text, summary, None


In [25]:
#7.Batch translation for all audio files
input_folder = "audio_input"
output_folder = "audio_output"
os.makedirs(output_folder, exist_ok=True)

for file_name in os.listdir(input_folder):
    if file_name.lower().endswith((".wav", ".mp3", ".m4a")):
        file_path = os.path.join(input_folder, file_name)
        print(f"\nProcessing file: {file_path}")

        translated_text, summary, audio_output = audio_to_translated(file_path, target_language="ta")

        if translated_text is None:
            print(f"Skipping {file_name} (failed)")
            continue

        base_name = os.path.splitext(file_name)[0]
        timestamp = int(time.time())
        output_path = os.path.join(output_folder, f"{base_name}_{timestamp}_translated.mp3")
        
        if summary and summary.get("audio_out_path"):
            os.rename(summary["audio_out_path"], output_path)
            summary["audio_out_path"] = output_path

        if translated_text:
            #print(f"\nTranslated Text:\n{translated_text}")
            print("\nSUMMARY:", json.dumps(summary, indent=2, ensure_ascii=False))
            display(audio_output)


Processing file: audio_input\puretone_wav.wav
Processing audio file: audio_input\puretone_wav.wav
Transcribed Text: This file contains a sequence of pure tones at various frequencies designed for audio system testing. Provided by samplefiles.com.
Translated Text: இந்த கோப்பு ஒலியியல் அமைப்பின் சோதனைக்காக வடிவமைக்கப்பட்ட பல்வேறு அதிர்வெண்களில் உள்ள தூய சத்தங்களின் வரிசையை கொண்டுள்ளது. samplefiles.com மூலம் வழங்கப்பட்டது.
Audio saved at: audio_output\translated_audio.mp3

SUMMARY: {
  "input_tokens": 19,
  "output_tokens": 16,
  "total_tokens": 35,
  "openai_cost_usd": 5e-06,
  "aws_cost_usd": 0.000636,
  "translated_text": "இந்த கோப்பு ஒலியியல் அமைப்பின் சோதனைக்காக வடிவமைக்கப்பட்ட பல்வேறு அதிர்வெண்களில் உள்ள தூய சத்தங்களின் வரிசையை கொண்டுள்ளது. samplefiles.com மூலம் வழங்கப்பட்டது.",
  "audio_out_path": "audio_output\\puretone_wav_1762409769_translated.mp3"
}



Processing file: audio_input\real_mp3.mp3
Processing audio file: audio_input\real_mp3.mp3
Transcribed Text: Oh, he has been away from New York. He has been all round the world. He doesn't know many people here, but he's very sociable, and he wants to know everyone.
Translated Text: Here is the translation of your text into Tamil:

"அவன் நியூயார்க் இருந்து தொலைவில் இருக்கிறான். அவன் உலகம் முழுவதும் சுற்றி வந்திருக்கிறான். இங்கு பலரை அவன் அறியவில்லை, ஆனால் அவன் மிகவும் சமூகமாக இருக்கிறான், மற்றும் அவன் அனைவரையும் அறிய விரும்புகிறான்."
Audio saved at: audio_output\translated_audio.mp3

SUMMARY: {
  "input_tokens": 31,
  "output_tokens": 33,
  "total_tokens": 64,
  "openai_cost_usd": 1e-05,
  "aws_cost_usd": 0.00102,
  "translated_text": "Here is the translation of your text into Tamil:  \"அவன் நியூயார்க் இருந்து தொலைவில் இருக்கிறான். அவன் உலகம் முழுவதும் சுற்றி வந்திருக்கிறான். இங்கு பலரை அவன் அறியவில்லை, ஆனால் அவன் மிகவும் சமூகமாக இருக்கிறான், மற்றும் அவன் அனைவரையும் அறிய விரும்புகிறான்


Processing file: audio_input\sample1_mp3.mp3
Processing audio file: audio_input\sample1_mp3.mp3
Transcribed Text: He doesn't belong to you, and I don't see how you have anything to do with what is be his power. He's he personified from this stage to you.
Translated Text: The text you provided seems to have some unclear phrases, which makes it difficult to translate accurately. However, I can provide a translation based on the general meaning. Here’s a possible translation in Tamil:

அவன் உனக்கு சொந்தமானவன் அல்ல, அவன் என்னுடைய சக்தியுடன் உனக்கு எவ்வாறு தொடர்பு இருக்கிறது என்று நான் பார்க்கவில்லை. அவன் இந்த மேடையில் இருந்து உனக்கு உருவாக்கப்பட்டவன்.

If you have any specific adjustments or clarifications, feel free to let me know!
Audio saved at: audio_output\translated_audio.mp3

SUMMARY: {
  "input_tokens": 29,
  "output_tokens": 68,
  "total_tokens": 97,
  "openai_cost_usd": 1.5e-05,
  "aws_cost_usd": 0.001864,
  "translated_text": "The text you provided seems to have some unclear ph

In [36]:
# 8. Example Test Run - Single File
audio_file = r"E:\HOPE\AI Course Tamil\translation_project\audio_input\pragathi_audio.mp3"

# Ask target language (optional)
target_language_usr = input("Enter target language code (like 'ta' for Tamil, 'fr' for French): ")
translated_text, summary, audio_output = audio_to_translated(audio_file, target_language=target_language_usr)

print("\nSUMMARY:", json.dumps(summary, indent=2, ensure_ascii=False))
display(audio_output)  # plays the translated speech inside notebook


Enter target language code (like 'ta' for Tamil, 'fr' for French):  fr


Processing audio file: E:\HOPE\AI Course Tamil\translation_project\audio_input\pragathi_audio.mp3
File too large (3.08 MB). Please use a file under 2 MB.

SUMMARY: null


None

In [13]:
# 8️⃣ Play translated audio
display(Audio(summary["audio_out_path"]))
print(summary.keys())

dict_keys(['input_tokens', 'output_tokens', 'total_tokens', 'openai_cost_usd', 'aws_cost_usd', 'translated_text', 'audio_out_path'])
